# A Shift/Reduce Parser for Arithmetic Expressions

F# counterpart of `ANTLR4-Python/LR-Parser-Generator/Shift-Reduce-Parser-Pure.ipynb`. This is a bottom-up parser. The original description incorrectly called it recursive descent and loaded a missing or incompatible `parse-table.py`; here the LR table is regenerated from the displayed arithmetic grammar.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#r "../../Chapter-08/GrammarSyntax/bin/Debug/net9.0/FsLexYacc.Runtime.dll"
#r "../../Chapter-08/GrammarSyntax/bin/Debug/net9.0/GrammarSyntax.dll"
#load "../../Chapter-09/SLR.fsx"
#load "../../Chapter-09/LR.fsx"
#load "../../Chapter-09/ParseTable.fsx"
#load "../../Chapter-09/ShiftReducePure.fsx"
open GrammarSyntax
open SLR
open ShiftReducePure


In [ ]:
let grammar = GrammarSyntax.Parse.file "arith.g"
let table = LR.computeTables grammar
checkNotebook (SLR.conflicts table |> List.isEmpty)
SLR.printTable LR.describeItem table

## Scanner and parser driver

Tokens preserve their lexemes. The state stack has one more entry than the semantic-value stack. Shift pushes a token; reduce pops a production body and follows a goto; accept requires the complete input. The same generic driver supports recognition and tree construction. This is its canonical implementation from Chapter 09:

```fsharp
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"

// The state stack has one more entry than the semantic-value stack.
let parseBy table leaf reduce tokens =
    let rec step states values remaining =
        match states, remaining with
        | state :: _, token :: tail ->
            match Map.tryFind (state, token.Kind) table.Actions with
            | Some candidates when Set.count candidates = 1 ->
                match Set.minElement candidates with
                | Shift target -> step (target :: states) (leaf token :: values) tail
                | Reduce rule ->
                    let count = rule.Body.Length
                    if count >= states.Length || count > values.Length then Error "Invalid reduction stack"
                    else
                        let children = values |> List.take count |> List.rev
                        let restStates, restValues = List.skip count states, List.skip count values
                        match Map.tryFind (restStates.Head, rule.Head) table.Gotos with
                        | Some target -> step (target :: restStates) (reduce rule children :: restValues) remaining
                        | None -> Error(sprintf "Missing goto for %s" rule.Head)
                | Accept ->
                    match values, remaining with
                    | [value], [last] when last.Kind = "$" -> Ok value
                    | _ -> Error "Invalid accepting configuration"
            | Some candidates -> Error(sprintf "Conflicting actions in s%d for %s: %A" state token.Kind candidates)
            | None -> Error(sprintf "Syntax error in s%d at token %s" state token.Lexeme)
        | _ -> Error "Incomplete parser configuration"
    step [0] [] (tokens @ [{ Kind = "$"; Lexeme = "end of input" }])

let parse table tokens =
    match parseBy table (fun _ -> ()) (fun _ _ -> ()) tokens with Ok () -> true | Error _ -> false

let test text =
    let accepted = parse ParseTable.table (tokenize text)
    printfn "%s: %b" text accepted
    accepted
```


In [ ]:
let test text =
    let tokens = tokenize text
    printfn "Tokens: %A" tokens
    let accepted = ShiftReducePure.parse table tokens
    printfn "%s: %b" text accepted
    accepted

checkNotebook (test "(1 + 2) * 3")
checkNotebook (test "1 * 2 + 3 * (4 - 5) / 2")
checkNotebook (test "11+22*(33-44)/(5-10*5/(4-3))")
checkNotebook (not (test "1+2*3-"))
let badCharacter = try tokenize "1@2" |> ignore; false with _ -> true
checkNotebook badCharacter